In [ ]:
import os
from pyspark.sql.functions import col, trim, lower

# 1. Định nghĩa đường dẫn đến Volume chứa 9 file của bạn
volume_path = "/Volumes/data_dev_olist/bronze/raw"

# 2. Danh sách 9 file tương ứng với 9 bảng bạn muốn tạo ở tầng Silver
# (Bạn hãy sửa lại chính xác tên file đuôi .csv đang có trong Volume của bạn)
files_to_process = [
    {"file_name": "olist_customers_dataset.csv", "table_name": "olist_customers", "key": "customer_id"},
    {"file_name": "olist_geolocation_dataset.csv", "table_name": "olist_geolocation", "key": "geolocation_zip_code_prefix"},
    {"file_name": "olist_order_items_dataset.csv", "table_name": "olist_order_items", "key": "order_id"},
    {"file_name": "olist_order_payments_dataset.csv", "table_name": "olist_order_payments", "key": "order_id"},
    {"file_name": "olist_order_reviews_dataset.csv", "table_name": "olist_order_reviews", "key": "review_id"},
    {"file_name": "olist_orders_dataset.csv", "table_name": "olist_orders", "key": "order_id"},
    {"file_name": "olist_products_dataset.csv", "table_name": "olist_products", "key": "product_id"},
    {"file_name": "olist_sellers_dataset.csv", "table_name": "olist_sellers", "key": "seller_id"},
    {"file_name": "product_category_name_translation.csv", "table_name": "product_category_name_translation", "key": "product_category_name"}
]

# 3. Vòng lặp tự động xử lý hàng loạt
for item in files_to_process:
    full_file_path = os.path.join(volume_path, item["file_name"])
    target_table = f"data_dev_olist.silver.{item['table_name']}"
    
    print(f"🔄 Đang xử lý file: {item['file_name']} -> {target_table}...")
    
    try:
        # Bước A: Đọc file thô từ Volume
        df_bronze = (spark.read
                     .format("csv")
                     .option("header", "true")
                     .option("inferSchema", "true") # Tự động nhận diện kiểu dữ liệu int, string...
                     .load(full_file_path))
        
        # Bước B: Làm sạch dữ liệu cơ bản (Xóa khoảng trắng thừa, xóa dòng trùng lặp dựa trên Khóa chính)
        # Tự động quét qua tất cả các cột dạng chuỗi để trim() khoảng trắng
        string_cols = [c for c, t in df_bronze.dtypes if t == "string"]
        df_clean = df_bronze
        for col_name in string_cols:
            df_clean = df_clean.withColumn(col_name, trim(col(col_name)))
            
        # Lọc trùng theo khóa chính (Primary Key) của từng bảng
        df_silver = df_clean.dropDuplicates([item["key"]])
        
        # Bước C: Ghi thẳng thành bảng Delta ở tầng Silver (Tự động format sang Delta)
        (df_silver.write
         .format("delta")
         .mode("overwrite") # Dùng overwrite để tạo mới hoặc làm sạch bảng cũ mỗi lần chạy lại
         .saveAsTable(target_table))
         
        print(f"✅ Đã xử lý xong bảng: {target_table}")
        
    except Exception as e:
        print(f"❌ Lỗi khi xử lý file {item['file_name']}: {str(e)}")

print("🎉 Hoàn thành xử lý toàn bộ 9 file!")